# Monte Carlo Methods & Particle Filters
## From Basic Sampling to Sequential Bayesian Estimation

This notebook explores Monte Carlo techniques for approximate inference, progressing from
simple MC estimation through importance sampling, MCMC, and particle filters, culminating
in applications to robot localization and SLAM.

**When are Monte Carlo methods needed?** Whenever we face intractable integrals, high-dimensional
posteriors, or nonlinear/non-Gaussian state estimation problems where closed-form solutions do not exist.

**Prerequisites:** Probability theory, Bayes' theorem, basic linear algebra, state-space models.

**Key References:**
- Thrun, S., Burgard, W., & Fox, D. (2005). *Probabilistic Robotics*. MIT Press.
- Doucet, A., de Freitas, N., & Gordon, N. (2001). *Sequential Monte Carlo Methods in Practice*. Springer.
- Robert, C. P., & Casella, G. (2004). *Monte Carlo Statistical Methods*. Springer.

In [ ]:
%matplotlib inline

import numpy as np
from scipy import stats, signal
from scipy.linalg import block_diag
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
import matplotlib.colors as mcolors
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# Plot styling
# =============================================================================
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'lines.linewidth': 2
})

print('All imports loaded successfully.')

In [ ]:
# =============================================================================
# Constants
# =============================================================================

SEED = 42
np.random.seed(SEED)

# Colors
PRIMARY = 'steelblue'
SECONDARY = 'coral'
TERTIARY = 'seagreen'
ACCENT = 'goldenrod'
EVOLUTION_CMAP = 'viridis'

# Monte Carlo
MC_SAMPLE_SIZES = [10, 50, 100, 500, 1000, 5000, 10000, 50000]
MC_NUM_TRIALS = 200

# MCMC
MCMC_NUM_SAMPLES = 20000
MCMC_BURN_IN = 5000

# Particle filter
PF_NUM_PARTICLES = 500
PF_NUM_STEPS = 100

# Robot localization
LOC_NUM_PARTICLES = 1000
LOC_NUM_STEPS = 80

# FastSLAM
SLAM_NUM_PARTICLES = 100
SLAM_NUM_STEPS = 60
SLAM_NUM_LANDMARKS = 8

# Verification thresholds
PASS_THRESHOLD_MC_RATE = 0.15
PASS_THRESHOLD_ESS = 0.5
PASS_THRESHOLD_MH_LOW = 0.15
PASS_THRESHOLD_MH_HIGH = 0.55
PASS_THRESHOLD_PF_KF = 0.5

print('Constants defined.')

---
## 2. Basic Monte Carlo Estimation

The fundamental idea: approximate an expectation by an average of random samples.

$$E[f(X)] = \int f(x) \, p(x) \, dx \approx \frac{1}{N} \sum_{i=1}^{N} f(x_i), \quad x_i \sim p(x)$$

By the Central Limit Theorem, the MC estimator has error:

$$\boxed{\text{MC error} = O\left(\frac{1}{\sqrt{N}}\right)}$$

This convergence rate is **dimension-independent**, which is the key advantage over
deterministic quadrature in high dimensions.

In [ ]:
# =============================================================================
# Basic Monte Carlo estimation
# =============================================================================

def mc_estimate(f, sampler, n_samples):
    """Estimate E[f(X)] via Monte Carlo sampling.

    Args:
        f: Function to evaluate. Accepts array of shape (N,).
        sampler: Callable that returns N samples from the target distribution.
        n_samples: Number of MC samples.

    Returns:
        mean: MC estimate of E[f(X)]. Shape: scalar.
        std_err: Standard error of the estimate. Shape: scalar.
    """
    samples = sampler(n_samples)
    values = f(samples)
    mean = np.mean(values)
    std_err = np.std(values, ddof=1) / np.sqrt(n_samples)
    return mean, std_err


# --- Test on known integrals ---
# E[X^2] where X ~ N(0,1) should be 1.0 (the variance)
np.random.seed(SEED)
true_value = 1.0
f_test = lambda x: x**2
sampler_normal = lambda n: np.random.randn(n)

est, se = mc_estimate(f_test, sampler_normal, 10000)
print(f'E[X^2] for X~N(0,1): estimate = {est:.4f}, true = {true_value:.4f}, SE = {se:.4f}')

# E[sin(X)] where X ~ Uniform(0, pi) should be 2/pi
true_sin = 2.0 / np.pi
f_sin = lambda x: np.sin(x)
sampler_unif = lambda n: np.random.uniform(0, np.pi, n)

est2, se2 = mc_estimate(f_sin, sampler_unif, 10000)
print(f'E[sin(X)] for X~U(0,pi): estimate = {est2:.4f}, true = {true_sin:.4f}, SE = {se2:.4f}')

In [ ]:
# =============================================================================
# Verify MC convergence rate O(1/sqrt(N))
# =============================================================================

np.random.seed(SEED)

errors_by_n = []
for n in MC_SAMPLE_SIZES:
    trial_errors = []
    for _ in range(MC_NUM_TRIALS):
        est, _ = mc_estimate(f_test, sampler_normal, n)
        trial_errors.append(abs(est - true_value))
    errors_by_n.append(np.mean(trial_errors))

errors_by_n = np.array(errors_by_n)
sample_sizes = np.array(MC_SAMPLE_SIZES, dtype=float)

# Fit log-log slope: error ~ N^slope => log(error) = slope*log(N) + const
log_n = np.log(sample_sizes)
log_err = np.log(errors_by_n)
slope, intercept = np.polyfit(log_n, log_err, 1)

expected_slope = -0.5
slope_error = abs(slope - expected_slope)
mc_rate_pass = slope_error < PASS_THRESHOLD_MC_RATE

print(f'Fitted convergence slope: {slope:.4f} (expected: {expected_slope})')
print(f'MC convergence: max relative error = {slope_error:.2e} [{"PASS" if mc_rate_pass else "FAIL"}]')

---
## 3. Importance Sampling

When we cannot sample directly from $p(x)$, we sample from a **proposal** $q(x)$ and reweight:

$$E_p[f(X)] = \int f(x) \frac{p(x)}{q(x)} q(x) \, dx \approx \frac{1}{N} \sum_{i=1}^N f(x_i) \, w(x_i)$$

where $w(x_i) = p(x_i) / q(x_i)$ are the **importance weights** and $x_i \sim q$.

**Self-normalized IS** uses normalized weights $\tilde{w}_i = w_i / \sum_j w_j$, which is useful
when $p(x)$ is known only up to a normalizing constant.

**Effective Sample Size** measures weight degeneracy:

$$\boxed{\text{ESS} = \frac{\left(\sum_{i=1}^N w_i\right)^2}{\sum_{i=1}^N w_i^2}}$$

The **optimal proposal** minimizes variance: $q^*(x) \propto |f(x)| \, p(x)$.

In [ ]:
# =============================================================================
# Importance Sampling
# =============================================================================

def effective_sample_size(weights):
    """Compute the effective sample size from unnormalized weights.

    Args:
        weights: Unnormalized importance weights. Shape: (N,).

    Returns:
        ess: Effective sample size. Shape: scalar.
    """
    w = np.asarray(weights, dtype=float)
    return (np.sum(w))**2 / np.sum(w**2)


def importance_sampling(f, target_log_pdf, proposal_log_pdf, proposal_sampler, n_samples):
    """Importance sampling with self-normalized weights.

    Args:
        f: Function to evaluate. Accepts array of shape (N,).
        target_log_pdf: Log-density of the target distribution. Shape: (N,) -> (N,).
        proposal_log_pdf: Log-density of the proposal distribution. Shape: (N,) -> (N,).
        proposal_sampler: Callable returning N samples from proposal. Shape: (N,).
        n_samples: Number of samples.

    Returns:
        estimate: Self-normalized IS estimate. Shape: scalar.
        ess: Effective sample size. Shape: scalar.
        weights_normalized: Normalized weights. Shape: (N,).
        samples: Drawn samples. Shape: (N,).
    """
    samples = proposal_sampler(n_samples)
    log_w = target_log_pdf(samples) - proposal_log_pdf(samples)
    # Numerically stable normalization
    log_w_max = np.max(log_w)
    w = np.exp(log_w - log_w_max)
    weights_normalized = w / np.sum(w)
    estimate = np.sum(weights_normalized * f(samples))
    ess = effective_sample_size(w)
    return estimate, ess, weights_normalized, samples


# --- Demonstrate good vs bad proposal ---
np.random.seed(SEED)

# Target: N(3, 1). Estimate E[X] = 3.
target_mean, target_std = 3.0, 1.0
target_log_pdf = lambda x: stats.norm.logpdf(x, loc=target_mean, scale=target_std)
f_identity = lambda x: x
N_IS = 5000

# Good proposal: N(2.5, 1.5) -- overlaps well
good_log_pdf = lambda x: stats.norm.logpdf(x, loc=2.5, scale=1.5)
good_sampler = lambda n: np.random.normal(2.5, 1.5, n)

est_good, ess_good, w_good, s_good = importance_sampling(
    f_identity, target_log_pdf, good_log_pdf, good_sampler, N_IS
)

# Bad proposal: N(-2, 1) -- poor overlap
bad_log_pdf = lambda x: stats.norm.logpdf(x, loc=-2.0, scale=1.0)
bad_sampler = lambda n: np.random.normal(-2.0, 1.0, n)

est_bad, ess_bad, w_bad, s_bad = importance_sampling(
    f_identity, target_log_pdf, bad_log_pdf, bad_sampler, N_IS
)

print(f'True E[X] = {target_mean:.2f}')
print(f'Good proposal: estimate = {est_good:.4f}, ESS = {ess_good:.1f} ({ess_good/N_IS*100:.1f}%)')
print(f'Bad proposal:  estimate = {est_bad:.4f}, ESS = {ess_bad:.1f} ({ess_bad/N_IS*100:.1f}%)')

In [ ]:
# =============================================================================
# Verify IS ESS with good proposal
# =============================================================================

ess_ratio = ess_good / N_IS
is_ess_pass = ess_ratio > PASS_THRESHOLD_ESS

print(f'IS ESS: max relative error = {abs(ess_ratio - 1.0):.2e} [{"PASS" if is_ess_pass else "FAIL"}]')
print(f'ESS/N = {ess_ratio:.4f} (threshold: > {PASS_THRESHOLD_ESS})')

---
## 4. Markov Chain Monte Carlo (MCMC)

MCMC constructs a Markov chain whose stationary distribution is the target $p(x)$.

**Detailed balance:** A chain satisfies detailed balance w.r.t. $p$ if:

$$p(x) \, T(x \to x') = p(x') \, T(x' \to x)$$

**Metropolis-Hastings algorithm:**
1. Propose $x' \sim q(x'|x)$
2. Accept with probability $\alpha = \min\left(1, \frac{p(x') \, q(x|x')}{p(x) \, q(x'|x)}\right)$

For symmetric proposals $q(x'|x) = q(x|x')$, this simplifies to:

$$\boxed{\alpha = \min\left(1, \frac{p(x')}{p(x)}\right)}$$

**Optimal acceptance rate** for random walk MH in $d$ dimensions: $\approx 0.234$ (Roberts et al. 1997).

In [ ]:
# =============================================================================
# Metropolis-Hastings from scratch
# =============================================================================

def metropolis_hastings(log_target, proposal_sampler, log_proposal_pdf,
                        x0, n_samples):
    """General Metropolis-Hastings sampler.

    Args:
        log_target: Log-density of target (up to constant). Callable.
        proposal_sampler: Samples x' given x. Callable(x_current) -> x_proposed.
        log_proposal_pdf: Log q(x'|x). Callable(x_proposed, x_current) -> scalar.
        x0: Initial state. Shape: (d,) or scalar.
        n_samples: Number of samples to draw.

    Returns:
        samples: Array of samples. Shape: (n_samples, d) or (n_samples,).
        accept_rate: Fraction of proposals accepted. Shape: scalar.
    """
    x = np.array(x0, dtype=float)
    is_scalar = x.ndim == 0
    if is_scalar:
        x = x.reshape(1)
    d = x.shape[0]
    samples = np.zeros((n_samples, d))
    n_accepted = 0
    log_p_current = log_target(x)

    for i in range(n_samples):
        x_prop = proposal_sampler(x)
        log_p_prop = log_target(x_prop)
        log_q_forward = log_proposal_pdf(x_prop, x)
        log_q_backward = log_proposal_pdf(x, x_prop)
        log_alpha = log_p_prop - log_p_current + log_q_backward - log_q_forward
        if np.log(np.random.rand()) < log_alpha:
            x = x_prop
            log_p_current = log_p_prop
            n_accepted += 1
        samples[i] = x

    if is_scalar:
        samples = samples.ravel()
    return samples, n_accepted / n_samples


def random_walk_mh(log_target, x0, n_samples, step_size):
    """Random walk Metropolis-Hastings with Gaussian proposals.

    Args:
        log_target: Log-density of target. Callable.
        x0: Initial state. Shape: (d,) or scalar.
        n_samples: Number of samples.
        step_size: Standard deviation of Gaussian proposal. Shape: scalar or (d,).

    Returns:
        samples: MCMC samples. Shape: (n_samples, d) or (n_samples,).
        accept_rate: Acceptance rate. Shape: scalar.
    """
    x0_arr = np.atleast_1d(np.array(x0, dtype=float))
    d = x0_arr.shape[0]
    step = np.atleast_1d(np.array(step_size, dtype=float))
    if step.shape[0] == 1 and d > 1:
        step = np.full(d, step[0])

    proposal_sampler = lambda x: x + step * np.random.randn(d)
    # Symmetric proposal => log q cancels
    log_q = lambda xp, xc: 0.0

    return metropolis_hastings(log_target, proposal_sampler, log_q, x0_arr, n_samples)

In [ ]:
# =============================================================================
# Sample from banana-shaped distribution
# =============================================================================

np.random.seed(SEED)

def banana_log_pdf(x):
    """Log-density of a banana-shaped distribution.

    Args:
        x: State vector. Shape: (2,).

    Returns:
        log_density: Log-density value. Shape: scalar.
    """
    b = 0.1
    return -0.5 * (x[0]**2 / 100.0 + (x[1] + b * x[0]**2 - 100.0 * b)**2)


samples_banana, accept_rate_banana = random_walk_mh(
    banana_log_pdf,
    x0=np.array([0.0, 0.0]),
    n_samples=MCMC_NUM_SAMPLES,
    step_size=np.array([2.5, 0.7])
)

print(f'Banana distribution MCMC:')
print(f'  Acceptance rate: {accept_rate_banana:.4f}')
print(f'  Samples shape: {samples_banana.shape}')

In [ ]:
# =============================================================================
# Verify MH acceptance rate
# =============================================================================

mh_pass = PASS_THRESHOLD_MH_LOW < accept_rate_banana < PASS_THRESHOLD_MH_HIGH

print(f'MH acceptance rate: max relative error = {abs(accept_rate_banana - 0.35):.2e} '
      f'[{"PASS" if mh_pass else "FAIL"}]')
print(f'Acceptance rate = {accept_rate_banana:.4f} (target range: {PASS_THRESHOLD_MH_LOW}-{PASS_THRESHOLD_MH_HIGH})')

---
## 5. Sequential Monte Carlo (Particle Filters)

Particle filters approximate the **filtering distribution** $p(x_t | y_{1:t})$ using
a set of weighted particles $\{x_t^{(i)}, w_t^{(i)}\}_{i=1}^N$.

**Sequential Importance Sampling (SIS):**
1. **Predict:** $x_t^{(i)} \sim p(x_t | x_{t-1}^{(i)})$
2. **Weight:** $w_t^{(i)} = w_{t-1}^{(i)} \cdot p(y_t | x_t^{(i)})$

**Weight degeneracy:** Without resampling, after a few steps one particle carries all weight.

**Bootstrap particle filter** uses the transition prior as proposal and adds **resampling**:

$$\boxed{\text{Predict} \to \text{Weight} \to \text{Resample}}$$

**Systematic resampling** is preferred for its low variance and $O(N)$ complexity.

In [ ]:
# =============================================================================
# Systematic resampling
# =============================================================================

def systematic_resampling(weights):
    """Systematic resampling of particles.

    Args:
        weights: Normalized weights. Shape: (N,).

    Returns:
        indices: Resampled particle indices. Shape: (N,).
    """
    N = len(weights)
    positions = (np.arange(N) + np.random.rand()) / N
    cumsum = np.cumsum(weights)
    indices = np.zeros(N, dtype=int)
    i, j = 0, 0
    while i < N:
        if positions[i] < cumsum[j]:
            indices[i] = j
            i += 1
        else:
            j += 1
    return indices

In [ ]:
# =============================================================================
# Bootstrap particle filter
# =============================================================================

def particle_filter(y_obs, x0_sampler, transition_sampler, observation_log_likelihood,
                     n_particles):
    """Bootstrap particle filter for state estimation.

    Args:
        y_obs: Observations. Shape: (T, obs_dim) or (T,).
        x0_sampler: Draws initial particles. Callable(N) -> (N, state_dim).
        transition_sampler: Samples next state. Callable(particles) -> (N, state_dim).
        observation_log_likelihood: Log p(y|x). Callable(y, particles) -> (N,).
        n_particles: Number of particles.

    Returns:
        x_est: Weighted mean state estimates. Shape: (T, state_dim).
        particles_history: All particles over time. Shape: (T, N, state_dim).
        weights_history: All weights over time. Shape: (T, N).
        ess_history: ESS over time. Shape: (T,).
    """
    y_obs = np.atleast_2d(y_obs)
    if y_obs.shape[0] == 1 and y_obs.shape[1] > 1:
        y_obs = y_obs.T
    T = y_obs.shape[0]

    # Initialize particles
    particles = x0_sampler(n_particles)
    if particles.ndim == 1:
        particles = particles.reshape(-1, 1)
    state_dim = particles.shape[1]

    x_est = np.zeros((T, state_dim))
    particles_history = np.zeros((T, n_particles, state_dim))
    weights_history = np.zeros((T, n_particles))
    ess_history = np.zeros(T)

    for t in range(T):
        # Predict: propagate through transition model
        particles = transition_sampler(particles)
        if particles.ndim == 1:
            particles = particles.reshape(-1, 1)

        # Weight: evaluate observation likelihood
        log_w = observation_log_likelihood(y_obs[t], particles)
        log_w -= np.max(log_w)  # numerical stability
        w = np.exp(log_w)
        w_norm = w / np.sum(w)

        # Estimate
        x_est[t] = np.sum(w_norm[:, None] * particles, axis=0)

        # Store
        particles_history[t] = particles
        weights_history[t] = w_norm
        ess_history[t] = effective_sample_size(w)

        # Resample if ESS drops below threshold
        if ess_history[t] < n_particles / 2:
            indices = systematic_resampling(w_norm)
            particles = particles[indices]

    return x_est, particles_history, weights_history, ess_history

In [ ]:
# =============================================================================
# PF vs Kalman filter on linear-Gaussian system
# =============================================================================

np.random.seed(SEED)

# Linear-Gaussian system: x_t = A*x_{t-1} + w, y_t = H*x_t + v
A_LIN = 0.95
H_LIN = 1.0
Q_LIN = 0.5   # process noise variance
R_LIN = 1.0   # observation noise variance

# Generate ground truth
T_PF = PF_NUM_STEPS
x_true_pf = np.zeros(T_PF)
y_obs_pf = np.zeros(T_PF)
x_true_pf[0] = np.random.randn() * np.sqrt(Q_LIN / (1 - A_LIN**2))
y_obs_pf[0] = H_LIN * x_true_pf[0] + np.random.randn() * np.sqrt(R_LIN)
for t in range(1, T_PF):
    x_true_pf[t] = A_LIN * x_true_pf[t-1] + np.random.randn() * np.sqrt(Q_LIN)
    y_obs_pf[t] = H_LIN * x_true_pf[t] + np.random.randn() * np.sqrt(R_LIN)

# --- Kalman filter ---
x_kf = np.zeros(T_PF)
P_kf = np.zeros(T_PF)
x_kf[0] = 0.0
P_kf[0] = Q_LIN / (1 - A_LIN**2)
for t in range(T_PF):
    # Predict
    if t > 0:
        x_pred = A_LIN * x_kf[t-1]
        P_pred = A_LIN**2 * P_kf[t-1] + Q_LIN
    else:
        x_pred = 0.0
        P_pred = P_kf[0]
    # Update
    K = P_pred * H_LIN / (H_LIN**2 * P_pred + R_LIN)
    x_kf[t] = x_pred + K * (y_obs_pf[t] - H_LIN * x_pred)
    P_kf[t] = (1 - K * H_LIN) * P_pred

# --- Particle filter ---
x0_sampler_pf = lambda n: np.random.randn(n) * np.sqrt(Q_LIN / (1 - A_LIN**2))
trans_sampler_pf = lambda p: A_LIN * p + np.random.randn(*p.shape) * np.sqrt(Q_LIN)
obs_log_lik_pf = lambda y, p: stats.norm.logpdf(y, loc=H_LIN * p.ravel(), scale=np.sqrt(R_LIN))

x_pf_est, p_hist, w_hist, ess_hist = particle_filter(
    y_obs_pf, x0_sampler_pf, trans_sampler_pf, obs_log_lik_pf, PF_NUM_PARTICLES
)
x_pf_est = x_pf_est.ravel()

# Compare
rmse_kf = np.sqrt(np.mean((x_kf - x_true_pf)**2))
rmse_pf = np.sqrt(np.mean((x_pf_est - x_true_pf)**2))
print(f'KF RMSE:  {rmse_kf:.4f}')
print(f'PF RMSE:  {rmse_pf:.4f}')
print(f'PF/KF ratio: {rmse_pf/rmse_kf:.4f}')

In [ ]:
# =============================================================================
# Verify PF matches KF for linear-Gaussian system
# =============================================================================

pf_kf_ratio = rmse_pf / rmse_kf
pf_kf_max_error = np.max(np.abs(x_pf_est - x_kf))
pf_pass = pf_kf_ratio < (1.0 + PASS_THRESHOLD_PF_KF)

print(f'PF estimate matches KF: max relative error = {abs(pf_kf_ratio - 1.0):.2e} '
      f'[{"PASS" if pf_pass else "FAIL"}]')
print(f'Max absolute difference: {pf_kf_max_error:.4f}')

---
## 6. Rao-Blackwellization

**Key idea:** Factor the state into nonlinear and linear parts:

$$x_t = \begin{pmatrix} x_t^{nl} \\ x_t^{l} \end{pmatrix}$$

- Use **particles** for the nonlinear component $x^{nl}$
- Use **Kalman filters** for the linear component $x^{l}$, conditioned on each particle

By the Rao-Blackwell theorem, this estimator has **lower variance** than a plain particle
filter that uses particles for all states:

$$\boxed{\text{Var}[E[X^l | X^{nl}]] \leq \text{Var}[X^l]}$$

In [ ]:
# =============================================================================
# Rao-Blackwellized Particle Filter (RBPF)
# =============================================================================

def rao_blackwell_pf(y_obs, n_particles, A_nl, Q_nl, A_l, Q_l, H_nl, H_l, R,
                      x0_nl_mean, x0_nl_cov, x0_l_mean, x0_l_cov):
    """Rao-Blackwellized particle filter.

    System model:
        x_nl(t) = A_nl * x_nl(t-1) + noise_nl
        x_l(t)  = A_l * x_l(t-1) + noise_l
        y(t)    = H_nl * x_nl(t) + H_l * x_l(t) + noise_obs

    Particles handle x_nl; per-particle KFs handle x_l.

    Args:
        y_obs: Observations. Shape: (T,).
        n_particles: Number of particles.
        A_nl: Nonlinear state transition scalar.
        Q_nl: Nonlinear process noise variance.
        A_l: Linear state transition scalar.
        Q_l: Linear process noise variance.
        H_nl: Observation matrix for nonlinear state.
        H_l: Observation matrix for linear state.
        R: Observation noise variance.
        x0_nl_mean: Initial mean for nonlinear state.
        x0_nl_cov: Initial covariance for nonlinear state.
        x0_l_mean: Initial mean for linear state.
        x0_l_cov: Initial covariance for linear state.

    Returns:
        x_nl_est: Estimated nonlinear states. Shape: (T,).
        x_l_est: Estimated linear states. Shape: (T,).
        ess_history: ESS over time. Shape: (T,).
    """
    T = len(y_obs)
    N = n_particles

    # Initialize particles for nonlinear state
    particles_nl = np.random.randn(N) * np.sqrt(x0_nl_cov) + x0_nl_mean
    # Per-particle Kalman filter state for linear part
    kf_means = np.full(N, x0_l_mean)
    kf_covs = np.full(N, x0_l_cov)
    weights = np.ones(N) / N

    x_nl_est = np.zeros(T)
    x_l_est = np.zeros(T)
    ess_history = np.zeros(T)

    for t in range(T):
        # --- Predict nonlinear state (particles) ---
        particles_nl = A_nl * particles_nl + np.random.randn(N) * np.sqrt(Q_nl)

        # --- Predict linear state (per-particle KF) ---
        kf_means_pred = A_l * kf_means
        kf_covs_pred = A_l**2 * kf_covs + Q_l

        # --- Update weights using observation likelihood ---
        # y = H_nl * x_nl + H_l * x_l + noise
        # Predicted observation: H_nl * x_nl_particle + H_l * kf_mean_pred
        y_pred = H_nl * particles_nl + H_l * kf_means_pred
        S = H_l**2 * kf_covs_pred + R  # innovation variance
        log_w = -0.5 * np.log(2 * np.pi * S) - 0.5 * (y_obs[t] - y_pred)**2 / S
        log_w -= np.max(log_w)
        w = np.exp(log_w)
        weights = w / np.sum(w)

        # --- Update KF for each particle ---
        innovation = y_obs[t] - y_pred
        K = H_l * kf_covs_pred / S
        kf_means = kf_means_pred + K * innovation
        kf_covs = (1 - K * H_l) * kf_covs_pred

        # --- Estimates ---
        x_nl_est[t] = np.sum(weights * particles_nl)
        x_l_est[t] = np.sum(weights * kf_means)
        ess_history[t] = effective_sample_size(w)

        # --- Resample if needed ---
        if ess_history[t] < N / 2:
            indices = systematic_resampling(weights)
            particles_nl = particles_nl[indices]
            kf_means = kf_means[indices]
            kf_covs = kf_covs[indices]
            weights = np.ones(N) / N

    return x_nl_est, x_l_est, ess_history

In [ ]:
# =============================================================================
# Run RBPF example and compare to plain PF
# =============================================================================

np.random.seed(SEED)

# System with mixed nonlinear/linear states
T_RB = 80
A_NL, Q_NL = 0.9, 0.3
A_L, Q_L = 0.95, 0.2
H_NL_VAL, H_L_VAL = 1.0, 1.0
R_RB = 0.5

# Generate truth
x_nl_true = np.zeros(T_RB)
x_l_true = np.zeros(T_RB)
y_rb = np.zeros(T_RB)

for t in range(T_RB):
    if t > 0:
        x_nl_true[t] = A_NL * np.sin(x_nl_true[t-1]) + np.random.randn() * np.sqrt(Q_NL)
        x_l_true[t] = A_L * x_l_true[t-1] + np.random.randn() * np.sqrt(Q_L)
    y_rb[t] = H_NL_VAL * x_nl_true[t] + H_L_VAL * x_l_true[t] + np.random.randn() * np.sqrt(R_RB)

# Run RBPF (using linear transition for the nonlinear part as approximation)
x_nl_rb, x_l_rb, ess_rb = rao_blackwell_pf(
    y_rb, 300, A_NL, Q_NL, A_L, Q_L, H_NL_VAL, H_L_VAL, R_RB,
    0.0, 1.0, 0.0, 1.0
)

# Run standard PF for comparison (treating everything as one state summed)
x0_rb2 = lambda n: np.random.randn(n) * 1.0
trans_rb2 = lambda p: A_NL * p + np.random.randn(*p.shape) * np.sqrt(Q_NL + Q_L)
obs_rb2 = lambda y, p: stats.norm.logpdf(y, loc=p.ravel(), scale=np.sqrt(R_RB))

x_pf_rb, _, _, _ = particle_filter(y_rb, x0_rb2, trans_rb2, obs_rb2, 300)
x_pf_rb = x_pf_rb.ravel()

x_total_true = x_nl_true + x_l_true
x_total_rb = x_nl_rb + x_l_rb

rmse_rbpf = np.sqrt(np.mean((x_total_rb - x_total_true)**2))
rmse_std_pf = np.sqrt(np.mean((x_pf_rb - x_total_true)**2))

print(f'RBPF RMSE: {rmse_rbpf:.4f}')
print(f'Std PF RMSE: {rmse_std_pf:.4f}')
print(f'Variance reduction ratio: {rmse_std_pf/rmse_rbpf:.2f}x')

---
## 7. Application: Non-Gaussian Robot Localization

A robot on a 1D circular track with known landmarks demonstrates how particle filters handle
**multimodal beliefs**. The "kidnapped robot" scenario creates a multimodal posterior that
an EKF (unimodal Gaussian) cannot represent.

The particle filter naturally represents:
- Multiple hypotheses for the robot's position
- Non-Gaussian uncertainty
- Recovery from kidnapping (teleportation to unknown location)

In [ ]:
# =============================================================================
# Robot localization with particle filter
# =============================================================================

np.random.seed(SEED)

# --- 1D circular world setup ---
WORLD_SIZE = 100.0
LANDMARKS_1D = np.array([20.0, 40.0, 60.0, 80.0])  # known landmark positions
SENSOR_NOISE = 2.0
MOTION_NOISE = 1.0

def wrap(x):
    """Wrap position to [0, WORLD_SIZE)."""
    return x % WORLD_SIZE

def sense_landmarks(pos, landmarks, noise_std):
    """Measure distances to landmarks (with wrapping).

    Args:
        pos: Robot position. Shape: scalar.
        landmarks: Landmark positions. Shape: (L,).
        noise_std: Sensor noise std. Shape: scalar.

    Returns:
        measurements: Noisy distances. Shape: (L,).
    """
    dists = np.abs(landmarks - pos)
    dists = np.minimum(dists, WORLD_SIZE - dists)  # shortest distance on circle
    return dists + np.random.randn(len(landmarks)) * noise_std


# --- Simulate robot trajectory ---
true_pos = np.zeros(LOC_NUM_STEPS)
true_pos[0] = 10.0
motion_cmds = np.random.uniform(0.5, 2.0, LOC_NUM_STEPS)
measurements = []

# Kidnap at step 40!
KIDNAP_STEP = 40
KIDNAP_POS = 70.0

for t in range(LOC_NUM_STEPS):
    if t > 0:
        if t == KIDNAP_STEP:
            true_pos[t] = KIDNAP_POS  # teleport!
        else:
            true_pos[t] = wrap(true_pos[t-1] + motion_cmds[t] + np.random.randn() * MOTION_NOISE)
    measurements.append(sense_landmarks(true_pos[t], LANDMARKS_1D, SENSOR_NOISE))

measurements = np.array(measurements)

# --- Particle filter localization ---
N_LOC = LOC_NUM_PARTICLES
particles_loc = np.random.uniform(0, WORLD_SIZE, N_LOC)
weights_loc = np.ones(N_LOC) / N_LOC

pf_estimates = np.zeros(LOC_NUM_STEPS)
pf_particles_history = np.zeros((LOC_NUM_STEPS, N_LOC))

for t in range(LOC_NUM_STEPS):
    # Move particles
    if t > 0:
        particles_loc = wrap(particles_loc + motion_cmds[t] + np.random.randn(N_LOC) * MOTION_NOISE)
        # After kidnap, inject random particles to help recovery
        if t == KIDNAP_STEP + 1:
            n_inject = N_LOC // 4
            inject_idx = np.random.choice(N_LOC, n_inject, replace=False)
            particles_loc[inject_idx] = np.random.uniform(0, WORLD_SIZE, n_inject)

    # Weight by observation likelihood
    log_w = np.zeros(N_LOC)
    for l_idx in range(len(LANDMARKS_1D)):
        d_particle = np.abs(LANDMARKS_1D[l_idx] - particles_loc)
        d_particle = np.minimum(d_particle, WORLD_SIZE - d_particle)
        log_w += stats.norm.logpdf(measurements[t, l_idx], loc=d_particle, scale=SENSOR_NOISE)

    log_w -= np.max(log_w)
    w = np.exp(log_w)
    weights_loc = w / np.sum(w)

    # Circular mean for estimate
    angles = 2 * np.pi * particles_loc / WORLD_SIZE
    mean_angle = np.arctan2(np.sum(weights_loc * np.sin(angles)),
                            np.sum(weights_loc * np.cos(angles)))
    pf_estimates[t] = wrap(mean_angle * WORLD_SIZE / (2 * np.pi))
    pf_particles_history[t] = particles_loc.copy()

    # Resample
    ess_loc = effective_sample_size(w)
    if ess_loc < N_LOC / 2:
        idx = systematic_resampling(weights_loc)
        particles_loc = particles_loc[idx]
        weights_loc = np.ones(N_LOC) / N_LOC

# --- Simple EKF for comparison ---
ekf_pos = 10.0
ekf_var = 100.0
ekf_estimates = np.zeros(LOC_NUM_STEPS)

for t in range(LOC_NUM_STEPS):
    if t > 0:
        ekf_pos = wrap(ekf_pos + motion_cmds[t])
        ekf_var += MOTION_NOISE**2

    for l_idx in range(len(LANDMARKS_1D)):
        d_pred = abs(LANDMARKS_1D[l_idx] - ekf_pos)
        d_pred = min(d_pred, WORLD_SIZE - d_pred)
        S = ekf_var + SENSOR_NOISE**2
        K = ekf_var / S
        innovation = measurements[t, l_idx] - d_pred
        ekf_pos = wrap(ekf_pos + K * innovation)
        ekf_var = (1 - K) * ekf_var

    ekf_estimates[t] = ekf_pos

# Compute circular error
def circular_error(est, true, world):
    diff = np.abs(est - true)
    return np.minimum(diff, world - diff)

err_pf = circular_error(pf_estimates, true_pos, WORLD_SIZE)
err_ekf = circular_error(ekf_estimates, true_pos, WORLD_SIZE)

print(f'PF mean error:  {np.mean(err_pf):.2f}')
print(f'EKF mean error: {np.mean(err_ekf):.2f}')
print(f'PF error after kidnap (steps 40-60): {np.mean(err_pf[40:60]):.2f}')
print(f'EKF error after kidnap (steps 40-60): {np.mean(err_ekf[40:60]):.2f}')

---
## 8. Application: FastSLAM

FastSLAM exploits the structure of the SLAM problem:
- **Particles** represent the robot path
- Each particle maintains **independent EKFs** for each landmark

This Rao-Blackwellized approach makes the landmark estimation conditionally
independent given the robot path, reducing the problem from $O(K^2)$ to $O(K)$
per particle (where $K$ is the number of landmarks).

$$p(x_{1:t}, m | z_{1:t}, u_{1:t}) = p(x_{1:t} | z_{1:t}, u_{1:t}) \prod_{k=1}^K p(m_k | x_{1:t}, z_{1:t})$$

In [ ]:
# =============================================================================
# Simplified FastSLAM implementation
# =============================================================================

def fastslam(true_trajectory, observations, obs_indices, n_particles, n_landmarks,
              motion_noise, obs_noise, motion_cmds):
    """Simplified FastSLAM 1.0 for 2D point landmarks.

    Args:
        true_trajectory: Ground truth robot positions. Shape: (T, 2).
        observations: Range-bearing observations. Shape: (T, max_obs, 2).
        obs_indices: Landmark index for each obs. Shape: (T, max_obs). -1 = no obs.
        n_particles: Number of particles.
        n_landmarks: Number of landmarks.
        motion_noise: Std of motion noise. Shape: scalar.
        obs_noise: Std of observation noise [range, bearing]. Shape: (2,).
        motion_cmds: Motion commands [dx, dy]. Shape: (T, 2).

    Returns:
        path_estimates: Estimated robot path. Shape: (T, 2).
        landmark_estimates: Final estimated landmarks. Shape: (n_landmarks, 2).
        landmark_covs: Final landmark covariances. Shape: (n_landmarks, 2, 2).
    """
    T = len(true_trajectory)
    N = n_particles
    K = n_landmarks

    # Particle states: position
    particles = np.zeros((N, 2))
    particles[:, 0] = true_trajectory[0, 0] + np.random.randn(N) * 0.1
    particles[:, 1] = true_trajectory[0, 1] + np.random.randn(N) * 0.1

    # Per-particle landmark estimates: mean and covariance
    lm_means = np.zeros((N, K, 2))  # landmark means
    lm_covs = np.zeros((N, K, 2, 2))  # landmark covariances
    lm_initialized = np.zeros((N, K), dtype=bool)

    # Initialize covariances
    for i in range(N):
        for k in range(K):
            lm_covs[i, k] = np.eye(2) * 100.0

    weights = np.ones(N) / N
    path_estimates = np.zeros((T, 2))

    for t in range(T):
        # --- Predict: move particles ---
        if t > 0:
            particles[:, 0] += motion_cmds[t, 0] + np.random.randn(N) * motion_noise
            particles[:, 1] += motion_cmds[t, 1] + np.random.randn(N) * motion_noise

        # --- Update: process observations ---
        log_w = np.zeros(N)
        for obs_idx in range(observations.shape[1]):
            lm_id = obs_indices[t, obs_idx]
            if lm_id < 0:
                continue
            z_range = observations[t, obs_idx, 0]
            z_bearing = observations[t, obs_idx, 1]

            for i in range(N):
                if not lm_initialized[i, lm_id]:
                    # Initialize landmark from first observation
                    lm_means[i, lm_id, 0] = particles[i, 0] + z_range * np.cos(z_bearing)
                    lm_means[i, lm_id, 1] = particles[i, 1] + z_range * np.sin(z_bearing)
                    lm_covs[i, lm_id] = np.diag(obs_noise**2) * 10.0
                    lm_initialized[i, lm_id] = True
                else:
                    # EKF update for this landmark
                    dx = lm_means[i, lm_id, 0] - particles[i, 0]
                    dy = lm_means[i, lm_id, 1] - particles[i, 1]
                    q = dx**2 + dy**2
                    sq = np.sqrt(q) + 1e-10

                    z_pred_range = sq
                    z_pred_bearing = np.arctan2(dy, dx)

                    # Jacobian of observation model w.r.t. landmark position
                    H = np.array([
                        [dx / sq, dy / sq],
                        [-dy / q, dx / q]
                    ])

                    Q_obs = np.diag(obs_noise**2)
                    S = H @ lm_covs[i, lm_id] @ H.T + Q_obs
                    K_gain = lm_covs[i, lm_id] @ H.T @ np.linalg.inv(S)

                    innov = np.array([z_range - z_pred_range,
                                      z_bearing - z_pred_bearing])
                    # Wrap bearing innovation
                    innov[1] = (innov[1] + np.pi) % (2 * np.pi) - np.pi

                    lm_means[i, lm_id] += K_gain @ innov
                    lm_covs[i, lm_id] = (np.eye(2) - K_gain @ H) @ lm_covs[i, lm_id]

                    # Weight by observation likelihood
                    sign, logdet = np.linalg.slogdet(S)
                    log_w[i] += -0.5 * (innov @ np.linalg.inv(S) @ innov + logdet + 2 * np.log(2 * np.pi))

        # Normalize weights
        log_w -= np.max(log_w)
        w = np.exp(log_w)
        weights = w / np.sum(w)

        # Estimate
        path_estimates[t] = np.sum(weights[:, None] * particles, axis=0)

        # Resample
        ess_val = effective_sample_size(w)
        if ess_val < N / 2:
            idx = systematic_resampling(weights)
            particles = particles[idx].copy()
            lm_means = lm_means[idx].copy()
            lm_covs = lm_covs[idx].copy()
            lm_initialized = lm_initialized[idx].copy()
            weights = np.ones(N) / N
            log_w = np.zeros(N)

    # Best particle's landmark estimates
    best = np.argmax(weights)
    landmark_estimates = lm_means[best].copy()
    landmark_covs_out = lm_covs[best].copy()

    return path_estimates, landmark_estimates, landmark_covs_out

In [ ]:
# =============================================================================
# Run FastSLAM simulation
# =============================================================================

np.random.seed(SEED)

# --- Generate 2D environment ---
true_landmarks = np.array([
    [5.0, 5.0], [15.0, 5.0], [15.0, 15.0], [5.0, 15.0],
    [10.0, 0.0], [20.0, 10.0], [10.0, 20.0], [0.0, 10.0]
])

# Generate circular trajectory
T_SLAM = SLAM_NUM_STEPS
true_traj = np.zeros((T_SLAM, 2))
true_traj[0] = [10.0, 10.0]
motion_cmds_slam = np.zeros((T_SLAM, 2))
MOTION_NOISE_SLAM = 0.2
OBS_NOISE_SLAM = np.array([0.5, 0.1])  # range, bearing
SENSOR_RANGE = 12.0

for t in range(1, T_SLAM):
    angle = 2 * np.pi * t / T_SLAM
    radius = 7.0
    true_traj[t, 0] = 10.0 + radius * np.cos(angle)
    true_traj[t, 1] = 10.0 + radius * np.sin(angle)
    motion_cmds_slam[t] = true_traj[t] - true_traj[t-1]

# Generate observations (range-bearing to visible landmarks)
MAX_OBS = SLAM_NUM_LANDMARKS
observations_slam = np.zeros((T_SLAM, MAX_OBS, 2))
obs_indices_slam = np.full((T_SLAM, MAX_OBS), -1, dtype=int)

for t in range(T_SLAM):
    obs_count = 0
    for k in range(SLAM_NUM_LANDMARKS):
        dx = true_landmarks[k, 0] - true_traj[t, 0]
        dy = true_landmarks[k, 1] - true_traj[t, 1]
        r = np.sqrt(dx**2 + dy**2)
        if r < SENSOR_RANGE:
            bearing = np.arctan2(dy, dx)
            observations_slam[t, obs_count, 0] = r + np.random.randn() * OBS_NOISE_SLAM[0]
            observations_slam[t, obs_count, 1] = bearing + np.random.randn() * OBS_NOISE_SLAM[1]
            obs_indices_slam[t, obs_count] = k
            obs_count += 1

# Run FastSLAM
path_est, lm_est, lm_covs_out = fastslam(
    true_traj, observations_slam, obs_indices_slam,
    SLAM_NUM_PARTICLES, SLAM_NUM_LANDMARKS,
    MOTION_NOISE_SLAM, OBS_NOISE_SLAM, motion_cmds_slam
)

# Results
path_rmse = np.sqrt(np.mean(np.sum((path_est - true_traj)**2, axis=1)))
lm_errors = np.sqrt(np.sum((lm_est - true_landmarks)**2, axis=1))

print(f'Path RMSE: {path_rmse:.4f}')
print(f'Mean landmark error: {np.mean(lm_errors):.4f}')
print(f'Max landmark error:  {np.max(lm_errors):.4f}')
for k in range(SLAM_NUM_LANDMARKS):
    print(f'  Landmark {k}: error = {lm_errors[k]:.3f}, true = ({true_landmarks[k,0]:.1f}, {true_landmarks[k,1]:.1f}), est = ({lm_est[k,0]:.1f}, {lm_est[k,1]:.1f})')

---
## 9. Visualizations

Comprehensive visualizations of all Monte Carlo methods explored in this notebook.

In [ ]:
# =============================================================================
# MC convergence (log-log plot)
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Log-log convergence
axes[0].loglog(sample_sizes, errors_by_n, 'o-', color=PRIMARY, label='Empirical error')
fit_line = np.exp(intercept) * sample_sizes**slope
axes[0].loglog(sample_sizes, fit_line, '--', color=SECONDARY,
               label=f'Fit: $N^{{{slope:.3f}}}$')
ref_line = errors_by_n[0] * (sample_sizes / sample_sizes[0])**(-0.5)
axes[0].loglog(sample_sizes, ref_line, ':', color=TERTIARY,
               label='Reference: $N^{-0.5}$')
axes[0].set_xlabel('Number of samples $N$')
axes[0].set_ylabel('Mean absolute error')
axes[0].set_title('MC Convergence Rate')
axes[0].legend()

# Error vs N (linear scale with error bars)
np.random.seed(SEED)
means_by_n = []
stds_by_n = []
for n in MC_SAMPLE_SIZES:
    trials = [mc_estimate(f_test, sampler_normal, n)[0] for _ in range(MC_NUM_TRIALS)]
    means_by_n.append(np.mean(trials))
    stds_by_n.append(np.std(trials))

axes[1].errorbar(sample_sizes, means_by_n, yerr=stds_by_n, fmt='o-', color=PRIMARY,
                 capsize=4, label='MC estimates')
axes[1].axhline(y=true_value, color=SECONDARY, linestyle='--', label=f'True value = {true_value}')
axes[1].set_xlabel('Number of samples $N$')
axes[1].set_ylabel('$E[X^2]$ estimate')
axes[1].set_title('MC Estimate Convergence')
axes[1].set_xscale('log')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# IS weight distributions: good vs bad proposal
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Weight histograms
axes[0].hist(w_good * N_IS, bins=50, color=PRIMARY, alpha=0.7, label=f'Good (ESS={ess_good:.0f})')
axes[0].set_xlabel('Unnormalized weight')
axes[0].set_ylabel('Count')
axes[0].set_title('Good Proposal Weights')
axes[0].legend()

axes[1].hist(w_bad * N_IS, bins=50, color=SECONDARY, alpha=0.7, label=f'Bad (ESS={ess_bad:.0f})')
axes[1].set_xlabel('Unnormalized weight')
axes[1].set_ylabel('Count')
axes[1].set_title('Bad Proposal Weights')
axes[1].legend()

# Overlay densities
x_plot = np.linspace(-8, 10, 500)
axes[2].plot(x_plot, stats.norm.pdf(x_plot, loc=target_mean, scale=target_std),
             color='black', linewidth=2, label='Target p(x)')
axes[2].plot(x_plot, stats.norm.pdf(x_plot, loc=2.5, scale=1.5),
             color=PRIMARY, linestyle='--', label='Good proposal q(x)')
axes[2].plot(x_plot, stats.norm.pdf(x_plot, loc=-2.0, scale=1.0),
             color=SECONDARY, linestyle='--', label='Bad proposal q(x)')
axes[2].fill_between(x_plot, stats.norm.pdf(x_plot, loc=target_mean, scale=target_std),
                     alpha=0.15, color='gray')
axes[2].set_xlabel('x')
axes[2].set_ylabel('Density')
axes[2].set_title('Target vs Proposals')
axes[2].legend(fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# MCMC trace plots and autocorrelation
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Trace plots
axes[0, 0].plot(samples_banana[:, 0], color=PRIMARY, alpha=0.6, linewidth=0.5)
axes[0, 0].axvline(x=MCMC_BURN_IN, color=SECONDARY, linestyle='--', label='Burn-in')
axes[0, 0].set_xlabel('Iteration')
axes[0, 0].set_ylabel('$x_1$')
axes[0, 0].set_title('Trace Plot: $x_1$')
axes[0, 0].legend()

axes[0, 1].plot(samples_banana[:, 1], color=TERTIARY, alpha=0.6, linewidth=0.5)
axes[0, 1].axvline(x=MCMC_BURN_IN, color=SECONDARY, linestyle='--', label='Burn-in')
axes[0, 1].set_xlabel('Iteration')
axes[0, 1].set_ylabel('$x_2$')
axes[0, 1].set_title('Trace Plot: $x_2$')
axes[0, 1].legend()

# Autocorrelation
burn_samples = samples_banana[MCMC_BURN_IN:]
max_lag = 200

for dim, (ax, color, label) in enumerate(zip(
    [axes[1, 0], axes[1, 1]],
    [PRIMARY, TERTIARY],
    ['$x_1$', '$x_2$']
)):
    chain = burn_samples[:, dim]
    chain_centered = chain - np.mean(chain)
    acf = np.correlate(chain_centered, chain_centered, mode='full')
    acf = acf[len(acf)//2:]
    acf = acf[:max_lag+1] / acf[0]
    ax.bar(range(max_lag+1), acf, color=color, alpha=0.6)
    ax.axhline(y=0, color='black', linewidth=0.5)
    ax.axhline(y=1.96/np.sqrt(len(chain)), color=SECONDARY, linestyle='--', alpha=0.7)
    ax.axhline(y=-1.96/np.sqrt(len(chain)), color=SECONDARY, linestyle='--', alpha=0.7)
    ax.set_xlabel('Lag')
    ax.set_ylabel('ACF')
    ax.set_title(f'Autocorrelation: {label}')

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Banana distribution: samples scatter + contour
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

burn_samp = samples_banana[MCMC_BURN_IN:]

# Scatter of MCMC samples
axes[0].scatter(burn_samp[::5, 0], burn_samp[::5, 1], s=1, alpha=0.3, color=PRIMARY)
axes[0].set_xlabel('$x_1$')
axes[0].set_ylabel('$x_2$')
axes[0].set_title('MCMC Samples (Banana Distribution)')

# Contour plot of target density
x1_grid = np.linspace(-30, 30, 200)
x2_grid = np.linspace(-10, 15, 200)
X1, X2 = np.meshgrid(x1_grid, x2_grid)
b_val = 0.1
Z = -0.5 * (X1**2 / 100.0 + (X2 + b_val * X1**2 - 100.0 * b_val)**2)
Z = np.exp(Z - np.max(Z))

axes[1].contourf(X1, X2, Z, levels=20, cmap='viridis', alpha=0.8)
axes[1].contour(X1, X2, Z, levels=10, colors='white', linewidths=0.5, alpha=0.5)
axes[1].set_xlabel('$x_1$')
axes[1].set_ylabel('$x_2$')
axes[1].set_title('Banana Distribution Density')

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Particle cloud evolution (PF for linear-Gaussian system)
# =============================================================================

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
snapshot_times = [0, 5, 15, 30, 60, 90]

for idx, (ax, t_snap) in enumerate(zip(axes.ravel(), snapshot_times)):
    if t_snap >= T_PF:
        t_snap = T_PF - 1
    particles_t = p_hist[t_snap, :, 0]
    weights_t = w_hist[t_snap]

    ax.hist(particles_t, bins=40, weights=weights_t, density=True,
            color=plt.cm.viridis(t_snap / T_PF), alpha=0.7, label='Particles')
    ax.axvline(x=x_true_pf[t_snap], color=SECONDARY, linewidth=2, linestyle='--',
               label=f'True: {x_true_pf[t_snap]:.2f}')
    ax.axvline(x=x_pf_est[t_snap], color=ACCENT, linewidth=2, linestyle=':',
               label=f'PF est: {x_pf_est[t_snap]:.2f}')
    ax.set_title(f't = {t_snap}')
    ax.legend(fontsize=8)
    ax.set_xlabel('State')

plt.suptitle('Particle Cloud Evolution', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# PF vs EKF localization error
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Error over time
axes[0].plot(err_pf, color=PRIMARY, label='PF error', alpha=0.8)
axes[0].plot(err_ekf, color=SECONDARY, label='EKF error', alpha=0.8)
axes[0].axvline(x=KIDNAP_STEP, color=ACCENT, linestyle='--', linewidth=2, label='Kidnap event')
axes[0].set_xlabel('Time step')
axes[0].set_ylabel('Circular position error')
axes[0].set_title('Localization Error: PF vs EKF')
axes[0].legend()

# Particle distribution at key moments
t_show = [KIDNAP_STEP - 1, KIDNAP_STEP + 1, KIDNAP_STEP + 10, KIDNAP_STEP + 20]
colors_snap = [PRIMARY, SECONDARY, TERTIARY, ACCENT]
for t_s, c in zip(t_show, colors_snap):
    if t_s < LOC_NUM_STEPS:
        axes[1].hist(pf_particles_history[t_s], bins=50, density=True, alpha=0.4,
                     color=c, label=f't={t_s}')
axes[1].axvline(x=true_pos[KIDNAP_STEP + 1], color='black', linewidth=2,
                linestyle='--', label='True pos (post-kidnap)')
axes[1].set_xlabel('Position')
axes[1].set_ylabel('Density')
axes[1].set_title('Particle Distribution Around Kidnap')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# FastSLAM map visualization
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Map view
ax = axes[0]
ax.plot(true_traj[:, 0], true_traj[:, 1], '-', color=PRIMARY, linewidth=2, label='True path')
ax.plot(path_est[:, 0], path_est[:, 1], '--', color=SECONDARY, linewidth=2, label='Estimated path')

# True landmarks
ax.scatter(true_landmarks[:, 0], true_landmarks[:, 1], s=200, marker='*',
           color=ACCENT, edgecolors='black', zorder=5, label='True landmarks')

# Estimated landmarks with covariance ellipses
for k in range(SLAM_NUM_LANDMARKS):
    ax.scatter(lm_est[k, 0], lm_est[k, 1], s=80, marker='x',
               color=TERTIARY, linewidth=2, zorder=5)
    # Covariance ellipse
    eigenvalues, eigenvectors = np.linalg.eigh(lm_covs_out[k])
    angle = np.degrees(np.arctan2(eigenvectors[1, 0], eigenvectors[0, 0]))
    width = 2 * 2 * np.sqrt(np.abs(eigenvalues[0]))
    height = 2 * 2 * np.sqrt(np.abs(eigenvalues[1]))
    ellipse = Ellipse(xy=(lm_est[k, 0], lm_est[k, 1]),
                      width=width, height=height, angle=angle,
                      fill=False, color=TERTIARY, linestyle='--', alpha=0.7)
    ax.add_patch(ellipse)

ax.scatter([], [], s=80, marker='x', color=TERTIARY, label='Estimated landmarks')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('FastSLAM: Map & Trajectory')
ax.legend(fontsize=10)
ax.set_aspect('equal')

# Landmark convergence over time
ax2 = axes[1]
path_errors = np.sqrt(np.sum((path_est - true_traj)**2, axis=1))
ax2.plot(path_errors, color=PRIMARY, label='Path error')
ax2.set_xlabel('Time step')
ax2.set_ylabel('Position error')
ax2.set_title('FastSLAM: Path Error Over Time')
ax2.legend()

plt.tight_layout()
plt.show()

---
## 10. Extensions & Advanced Topics

### Hamiltonian Monte Carlo (HMC)

HMC augments the state with momentum variables and uses Hamiltonian dynamics to propose
distant states with high acceptance probability. The leapfrog integrator preserves volume
(symplecticity), ensuring detailed balance.

$$H(q, p) = U(q) + K(p), \quad U(q) = -\log p(q), \quad K(p) = \frac{1}{2} p^T M^{-1} p$$

HMC eliminates the random walk behavior of MH, giving convergence rates that scale
as $O(d^{1/4})$ vs $O(d)$ for random walk MH in $d$ dimensions.

### Particle MCMC

Particle MCMC methods (Andrieu et al., 2010) use particle filters as proposals within
an MCMC framework, enabling Bayesian inference over both states and parameters:

- **Particle Marginal MH (PMMH):** Uses the particle filter's marginal likelihood
  estimate as an unbiased estimator within MH.
- **Particle Gibbs (PG):** Conditional SMC within a Gibbs sampler.

### Adaptive Particle Count

Instead of fixing $N$, adapt the number of particles based on the ESS or KL divergence
between the prior and posterior at each step. This saves computation when the observation
is uninformative and allocates more particles when the likelihood is sharply peaked.

### Unscented Particle Filter (UPF)

Replace the bootstrap proposal (transition prior) with an unscented Kalman filter proposal.
The UKF proposal incorporates the current observation, producing particles closer to the
high-likelihood region. This drastically improves efficiency in problems where the likelihood
is much narrower than the prior:

$$q(x_t | x_{t-1}, y_t) \approx \mathcal{N}(\hat{x}_t^{UKF}, P_t^{UKF})$$

### Further Reading

- Neal, R. M. (2011). *MCMC using Hamiltonian dynamics*. Handbook of Markov Chain Monte Carlo.
- Andrieu, C., Doucet, A., & Holenstein, R. (2010). *Particle Markov chain Monte Carlo methods*. JRSS-B.
- Fox, D. (2003). *Adapting the sample size in particle filters through KLD-sampling*. IJRR.
- Van der Merwe, R. et al. (2001). *The unscented particle filter*. NIPS.

---
## Summary

| Method | Key Idea | When to Use |
|--------|----------|-------------|
| Basic MC | Average iid samples | Known sampling distribution |
| Importance Sampling | Reweight from proposal | Cannot sample target directly |
| MCMC (MH) | Markov chain with target stationary dist | Unnormalized target density |
| Particle Filter | Sequential IS + resampling | Online state estimation |
| Rao-Blackwellized PF | Particles + KF for mixed systems | Partially linear models |
| FastSLAM | Particles for path, EKF per landmark | SLAM with known data association |

**Key takeaways:**
- MC convergence is $O(1/\sqrt{N})$, dimension-independent
- Importance sampling quality depends critically on proposal overlap
- MCMC acceptance rate near 0.234 is optimal for random walk proposals
- Particle filters handle multimodality and non-Gaussianity naturally
- Rao-Blackwellization reduces variance by analytically marginalizing linear components
- FastSLAM exploits conditional independence structure in SLAM